# 16.100 Project 1 — starter notebook

This notebook gets you set up for Tasks 2 and 3. It shows you how to

1. keep every handout value in **`p1_inputs.toml`** and read it into Python, so you never type a handout number into your code;
2. write functions that use those values;
3. fill in **`p1_answers.toml`** and check its format before you submit.

**Files in this folder**

| File | What it is | Do you edit it? |
|---|---|---|
| `p1_starter.ipynb` | this notebook | yes: this is your work |
| `p1_inputs.toml` | every value given in the handout, and your design choices in `[design]` | only `[design]` (and placeholders in later projects) |
| `p1_io.py` | reads `p1_inputs.toml` into a Python object | no |
| `p1_answers.toml` | the answer file you submit | yes: replace every `nan` |
| `validate_p1_answers.py` | checks the format of your answer file | no |

**Running in the browser.** If you opened this at the course's JupyterLite link, everything runs inside your browser (Python via Pyodide); nothing is installed and nothing is sent to a server. Your edits are saved **in this browser only**. Download your files regularly (right-click a file in the file browser on the left, then *Download*), and download them before you clear your browser data or switch computers.

**Running on your own computer.** Download the folder and open the notebook with Jupyter (Python 3.11 or later). Nothing else is needed.

**What to upload to Canvas:** your report PDF, this notebook (and any other notebooks or scripts you used), and your filled-in `p1_answers.toml`.

## 1. Read the handout values

`load_inputs` reads `p1_inputs.toml` and returns an object whose fields are the keys of the file: `inp.rho`, `inp.CD0`, `inp.P_max_W`, and so on. Open `p1_inputs.toml` (double-click it in the file browser) to see every name.

In [5]:
import math
from P1_p1_io import load_inputs, require_design

inp = load_inputs("P1_p1_inputs.toml")
print("C_D0  =", inp.CD0)
print("C_Lmax =", inp.CLmax)
print("e*    =", inp.e_star_Whkg, "Wh/kg =", inp.e_star, "J/kg")
print("span limit =", inp.b_max_m, "m, stall speed limit =", inp.V_stall_max, "m/s")

C_D0  = 0.035
C_Lmax = 1.4
e*    = 150.0 Wh/kg = 540000.0 J/kg
span limit = 1.6 m, stall speed limit = 7.0 m/s


## 2. Functions that use the values

Pass `inp` into every function, and take every number from it. Two worked examples:

In [ ]:
def dynamic_pressure(inp, V):
    """q = 1/2 rho V^2, in Pa."""
    return 0.5 * inp.rho * V**2


def reynolds_number(inp, V, chord):
    """Re = V c / nu, based on the chord."""
    return V * chord / inp.nu


V, chord = 20.0, 0.25   # picked only to show the functions working
print(f"q  = {dynamic_pressure(inp, V):.1f} Pa")
print(f"Re = {reynolds_number(inp, V, chord):.3g}")

## 3. Task 2: your turn

Fill in these functions from the equations in the handout. Each one should take its numbers from `inp`. The cell after them uses them for one design; extend it to the table Task 2 asks for.

In [ ]:
def wing_mass(inp, b, c):
    """Mass of the foam wing (kg) for span b and chord c (m)."""
    return inp.rho_foam*b*inp.g*inp.A_fac*c**2*inp.tau


def fuselage_mass(inp, b, S):
    """Fuselage mass (kg): everything except the wing, the battery and the payload."""
    return inp.m_fuse0_kg+inp.m_fuse_b_kg*b/inp.b0_m+inp.m_fuse_S_kg*S/inp.S0_m2


def stall_speed(inp, W, S):
    """Stall speed (m/s) for weight W (N) and wing area S (m^2)."""
    return math.sqrt((2*W)/(inp.rho*S*inp.CLmax))

In [7]:
S, AR, m_b = 0.40, 6.0, 0.30          # one example design; Task 2 needs a table of them


def calculate_all(inp, S, AR, m_b):

    b = math.sqrt(S * AR)
    c = S / b
    W_m = wing_mass(inp, b, c)
    F_m = fuselage_mass(inp, b, S)
    m = inp.payload_kg + W_m + F_m + m_b
    W = m*9.81
    wing_load = W/S
    V_s = stall_speed(inp, m * inp.g, S)

    return {"b": b, "c": c, "W_m": W_m, "F_m": F_m, "m": m, "W": W, "wing_load": wing_load, "V_s": V_s}

table = {}

S_range = [0.025, 0.2, 0.64] #calculated based off of a range for b [0.5, 1.6]
AR_range = [4, 7, 10] #decided based of a search (add reasoning later) saying 4-10 is a reasonable range for RC planes
m_b_range = [0.2, 0.4, 0.6]
for S in S_range:
    AR_dict = {}
    for AR in AR_range:
        m_b_dict = {}
        for m_b in m_b_range:
            run = calculate_all(inp, S, AR, m_b)
            m_b_dict[m_b] = run
        AR_dict[AR] = m_b_dict
    table[S] = AR_dict


{'b': 1.5491933384829668, 'c': 0.25819888974716115, 'W_m': 1, 'F_m': 1, 'm': 2.8, 'W': 27.468, 'wing_load': 68.67, 'V_s': 1}


## 4. Your design point (Task 3)

When you have chosen a design, put it in the `[design]` section of `p1_inputs.toml` (replace each `nan`), save the file, and run the cell below. It stops with a message until the design is filled in.

In [ ]:
inp = load_inputs("p1_inputs.toml")   # re-read, in case you edited the file
require_design(inp)

b = math.sqrt(inp.S_m2 * inp.AR)
c = inp.S_m2 / b
print(f"S = {inp.S_m2} m^2, AR = {inp.AR}, m_b = {inp.m_b_kg} kg, V = {inp.V_cruise} m/s")
print(f"b = {b:.3f} m, c = {c:.3f} m, cruise Re = {reynolds_number(inp, inp.V_cruise, c):.3g}")

## 5. Check your answer file

Open `p1_answers.toml`, replace every `nan` with your number, fill in `[team]`, and save. Then run the cell below. It checks the **format** only (every entry present, a number, no `nan` left), not whether your numbers are right. Fix what it lists and run it again until it says *format OK*.

In [ ]:
from validate_p1_answers import main as validate

validate(["validate_p1_answers.py", "p1_answers.toml"]);